# FaithTrace : GPU Inference Demo

This notebook demonstrates the NVIDIA inference optimization pipeline:
1. GPU auto-profiling (pynvml)
2. PyTorch DistilBERT failure classifier training
3. ONNX export with dynamic batch axes
4. TensorRT engine conversion (FP32 / FP16 / INT8)
5. Inference benchmarking (p50/p95/p99 latency + throughput)

**Runtime → Change runtime type → T4 GPU** before running.

## 1. Setup

In [1]:
# Verify GPU is available
!nvidia-smi

Sun May 31 21:04:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Clone the repo
!git clone https://github.com/sakshiasati17/FaithTrace.git
%cd FaithTrace
!git pull origin sakshi/main

fatal: destination path 'FaithTrace' already exists and is not an empty directory.
/content/FaithTrace
From https://github.com/sakshiasati17/FaithTrace
 * branch            sakshi/main -> FETCH_HEAD
Already up to date.


In [3]:
# Install core dependencies
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers onnx onnxruntime-gpu pynvml numpy onnxscript

In [4]:
# Add backend to path
import sys
sys.path.insert(0, 'backend')

## 2. GPU Auto-Profiling

In [5]:
from app.optimization.gpu_profiler import GPUAutoProfiler

profiler = GPUAutoProfiler()
profile = profiler.profile()

print(f"GPU: {profile.name}")
print(f"VRAM Total: {profile.vram_total_gb} GB")
print(f"VRAM Available: {profile.vram_available_gb} GB")
print(f"Compute Capability: {profile.compute_capability}")
print(f"Recommended Precision: {profile.recommended_precision}")
print(f"Max Batch Size: {profile.max_batch_size}")
print(f"\nReasoning: {profile.reasoning}")

GPU: Tesla T4
VRAM Total: 15.0 GB
VRAM Available: 14.6 GB
Compute Capability: (7, 5)
Recommended Precision: fp16
Max Batch Size: 16

Reasoning: Volta/Turing GPU (compute 7.5). FP16 tensor cores available with moderate batch size for 14.6 GB VRAM.


/content/FaithTrace/backend/app/optimization/gpu_profiler.py:32: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml


## 3. PyTorch DistilBERT Failure Classifier

In [6]:
import torch
from app.models.failure_classifier import FailureClassifier
from app.models.failure_classifier.dataset import LABEL_NAMES

model = FailureClassifier(num_classes=6)
model.eval()
model.cuda()

print(f"Model: DistilBERT encoder + RAGAS score fusion + 6-class head")
print(f"Classes: {LABEL_NAMES}")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Frozen params: {sum(p.numel() for p in model.parameters() if not p.requires_grad):,}")
print(f"Device: {next(model.parameters()).device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: DistilBERT encoder + RAGAS score fusion + 6-class head
Classes: ['no_failure', 'retrieval_miss', 'context_insufficient', 'hallucination', 'prompt_weakness', 'ranking_failure']
Total params: 66,563,078
Trainable params: 38,211,590
Frozen params: 28,351,488
Device: cuda:0


In [7]:
# Test forward pass with dummy data
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

sample_queries = [
    "What is the procurement threshold for emergency purchases?",
    "Show the Q1 revenue breakdown by region from the financial report",
    "What changed in version 3 of the vendor onboarding policy?",
    "What are the safety inspection intervals for heavy equipment?",
]

tokens = tokenizer(sample_queries, padding=True, truncation=True, max_length=128, return_tensors='pt')
input_ids = tokens['input_ids'].cuda()
attention_mask = tokens['attention_mask'].cuda()

# Dummy RAGAS scores (faithfulness, context_recall, context_precision, answer_relevance, answer_correctness)
ragas_scores = torch.tensor([
    [0.85, 0.72, 0.90, 0.88, 0.76],
    [0.20, 0.15, 0.30, 0.45, 0.22],  # likely table_retrieval_miss
    [0.40, 0.60, 0.55, 0.70, 0.35],  # likely context_insufficient
    [0.95, 0.88, 0.92, 0.90, 0.91],  # likely no_failure
], dtype=torch.float32).cuda()

with torch.no_grad():
    logits = model(input_ids, attention_mask, ragas_scores)
    preds = torch.argmax(logits, dim=1)

print("\nPredictions (untrained model — random):")
for q, p in zip(sample_queries, preds):
    print(f"  {q[:60]}...  →  {LABEL_NAMES[p.item()]}")

print(f"\nLogits shape: {logits.shape}")
print(f"Softmax probabilities:\n{torch.softmax(logits, dim=1).cpu().numpy().round(3)}")


Predictions (untrained model — random):
  What is the procurement threshold for emergency purchases?...  →  no_failure
  Show the Q1 revenue breakdown by region from the financial r...  →  retrieval_miss
  What changed in version 3 of the vendor onboarding policy?...  →  retrieval_miss
  What are the safety inspection intervals for heavy equipment...  →  prompt_weakness

Logits shape: torch.Size([4, 6])
Softmax probabilities:
[[0.202 0.187 0.159 0.131 0.178 0.143]
 [0.175 0.231 0.134 0.15  0.15  0.16 ]
 [0.165 0.219 0.15  0.117 0.197 0.153]
 [0.178 0.182 0.163 0.128 0.199 0.15 ]]


In [8]:
# # Install core dependencies
# !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
# !pip install -q transformers onnx onnxruntime-gpu pynvml numpy onnxscript

In [10]:
import os
import onnx
os.makedirs('models', exist_ok=True)
onnx_path = 'models/failure_classifier.onnx'

model.cpu().eval()
dummy_ids   = torch.randint(0, 30522, (1, 512))
dummy_mask  = torch.ones(1, 512, dtype=torch.long)
dummy_ragas = torch.rand(1, 5)

with torch.no_grad():
    traced = torch.jit.trace(model, (dummy_ids, dummy_mask, dummy_ragas), strict=False)

torch.onnx.export(
    traced,
    (dummy_ids, dummy_mask, dummy_ragas),
    onnx_path,
    dynamo=False,
    opset_version=14,
    export_params=True,
    do_constant_folding=True,
    input_names=["input_ids", "attention_mask", "ragas_scores"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids":      {0: "batch_size"},
        "attention_mask": {0: "batch_size"},
        "ragas_scores":   {0: "batch_size"},
        "logits":         {0: "batch_size"},
    },
)

onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print(f"ONNX exported: {onnx_path}")
print(f"File size: {os.path.getsize(onnx_path) / 1024 / 1024:.1f} MB")

/tmp/ipykernel_22117/2760984114.py:14: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:1510: UserWarning: no signature found for builtin <built-in method __call__ of pybind11_builtins.pybind11_detail_function_record_v1_system_libstdcpp_gxx_abi_1xxx_use_cxx11_abi_1 object at 0x7e4be283f750>, skipping _decide_input_format
  args = _decide_input_format(model, args)
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 14 is achi

ONNX exported: models/failure_classifier.onnx
File size: 254.0 MB


## 4. ONNX Export

In [11]:
# Validate ONNX model
import onnx

onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("ONNX model validation: PASSED")
print(f"IR version: {onnx_model.ir_version}")
print(f"Opset: {onnx_model.opset_import[0].version}")
print(f"\nInputs:")
for inp in onnx_model.graph.input:
    print(f"  {inp.name}: {[d.dim_param or d.dim_value for d in inp.type.tensor_type.shape.dim]}")
print(f"\nOutputs:")
for out in onnx_model.graph.output:
    print(f"  {out.name}: {[d.dim_param or d.dim_value for d in out.type.tensor_type.shape.dim]}")

ONNX model validation: PASSED
IR version: 7
Opset: 14

Inputs:
  input_ids: ['batch_size', 512]
  attention_mask: ['batch_size', 512]
  ragas_scores: ['batch_size', 5]

Outputs:
  logits: ['batch_size', 6]


## 5. ONNX Runtime Inference Comparison

In [12]:
import onnxruntime as ort
import numpy as np
import time

# ONNX was traced with batch_size=1. DistilBERT's internal attention reshapes
# get baked in at trace time — dynamic_axes only covers outer inputs, not
# internal ops. Running batch_size=1 avoids the reshape mismatch.
# For throughput, both backends are benchmarked at batch_size=1 (per-query).
tokenizer_onnx = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
tokens_single = tokenizer_onnx(
    [sample_queries[0]], padding='max_length', truncation=True, max_length=512, return_tensors='pt'
)
input_ids_np   = tokens_single['input_ids'].numpy()       # (1, 512)
attn_mask_np   = tokens_single['attention_mask'].numpy()  # (1, 512)
ragas_np       = ragas_scores[:1].cpu().numpy()            # (1, 5)

# Setup ONNX Runtime with GPU
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
session = ort.InferenceSession(onnx_path, providers=providers)
print(f"ONNX Runtime provider: {session.get_providers()[0]}")

def run_ort():
    return session.run(None, {
        'input_ids':      input_ids_np,
        'attention_mask': attn_mask_np,
        'ragas_scores':   ragas_np,
    })

# Prepare batch_size=1 PyTorch inputs for fair comparison
input_ids_pt = tokens_single['input_ids'].cuda()
attn_mask_pt = tokens_single['attention_mask'].cuda()
ragas_pt     = ragas_scores[:1]

def run_pt():
    with torch.no_grad():
        return model(input_ids_pt, attn_mask_pt, ragas_pt)

# Warmup
model.cuda().eval()
for _ in range(10):
    run_ort()
for _ in range(10):
    run_pt()
torch.cuda.synchronize()

# Benchmark
n_runs = 100
ort_times, pt_times = [], []

for _ in range(n_runs):
    start = time.perf_counter()
    run_ort()
    ort_times.append((time.perf_counter() - start) * 1000)

for _ in range(n_runs):
    torch.cuda.synchronize()
    start = time.perf_counter()
    run_pt()
    torch.cuda.synchronize()
    pt_times.append((time.perf_counter() - start) * 1000)

ort_times = sorted(ort_times)
pt_times  = sorted(pt_times)

print(f"\n{'='*60}")
print(f"  INFERENCE BENCHMARK (batch_size=1, {n_runs} runs)")
print(f"{'='*60}")
print(f"{'Metric':<20} {'PyTorch':>12} {'ONNX Runtime':>14} {'Speedup':>10}")
print(f"{'-'*60}")
for label, idx in [('p50 (ms)', n_runs//2), ('p95 (ms)', int(n_runs*0.95)), ('p99 (ms)', int(n_runs*0.99))]:
    pt_val  = pt_times[idx]
    ort_val = ort_times[idx]
    speedup = pt_val / ort_val if ort_val > 0 else 0
    print(f"{label:<20} {pt_val:>10.2f}ms {ort_val:>12.2f}ms {speedup:>9.1f}x")

pt_qps  = 1000 / np.mean(pt_times)
ort_qps = 1000 / np.mean(ort_times)
print(f"{'Throughput':<20} {pt_qps:>10.0f}qps {ort_qps:>12.0f}qps {ort_qps/pt_qps:>9.1f}x")
print(f"{'='*60}")

ONNX Runtime provider: CUDAExecutionProvider

  INFERENCE BENCHMARK (batch_size=1, 100 runs)
Metric                    PyTorch   ONNX Runtime    Speedup
------------------------------------------------------------
p50 (ms)                  20.26ms        20.95ms       1.0x
p95 (ms)                  21.18ms        21.93ms       1.0x
p99 (ms)                  21.60ms        23.73ms       0.9x
Throughput                   49qps           48qps       1.0x


## 6. TensorRT Conversion (if available)

In [13]:
# Try installing TensorRT (may not work on all Colab instances)
!pip install -q tensorrt 2>/dev/null || echo "TensorRT not available in this Colab runtime"

In [15]:
try:
    import tensorrt as trt
    TRT_AVAILABLE = True
except ImportError:
    TRT_AVAILABLE = False

if TRT_AVAILABLE:
    print(f"TensorRT {trt.__version__} available! Converting...")
    os.makedirs('models/tensorrt', exist_ok=True)
    TRT_LOGGER = trt.Logger(trt.Logger.WARNING)

    def build_engine(onnx_path, output_path, precision="fp32"):
        builder = trt.Builder(TRT_LOGGER)
        try:
            network = builder.create_network(
                1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
            )
        except AttributeError:
            network = builder.create_network()

        parser = trt.OnnxParser(network, TRT_LOGGER)
        with open(onnx_path, 'rb') as f:
            if not parser.parse(f.read()):
                errs = [str(parser.get_error(i)) for i in range(parser.num_errors)]
                raise RuntimeError(f"Parse failed: {errs}")

        config = builder.create_builder_config()
        config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 4 << 30)

        # TRT 10 removed BuilderFlag.FP16 / INT8 — guard with hasattr
        if precision == "fp16":
            if hasattr(trt.BuilderFlag, 'FP16'):
                config.set_flag(trt.BuilderFlag.FP16)
            else:
                print(f"  Note: TRT {trt.__version__} removed FP16 flag — building FP32")
        elif precision == "int8":
            if hasattr(trt.BuilderFlag, 'INT8'):
                config.set_flag(trt.BuilderFlag.INT8)
            else:
                print(f"  Note: TRT {trt.__version__} removed INT8 flag — building FP32")

        # Dynamic axes require an optimization profile.
        # Model traced at batch=1 so internal ops are fixed — set min=opt=max=1.
        profile = builder.create_optimization_profile()
        for i in range(network.num_inputs):
            inp = network.get_input(i)
            shape = [1 if d == -1 else d for d in inp.shape]
            profile.set_shape(inp.name, shape, shape, shape)
        config.add_optimization_profile(profile)

        serialized = builder.build_serialized_network(network, config)
        if serialized is None:
            raise RuntimeError("Engine build returned None")
        with open(output_path, 'wb') as f:
            f.write(serialized)
        return output_path

    for precision in ["fp32", "fp16", "int8"]:
        out = f"models/tensorrt/failure_classifier_{precision}.engine"
        try:
            build_engine(onnx_path, out, precision)
            print(f"  {precision}: {out} ({os.path.getsize(out)/1024/1024:.1f} MB)")
        except Exception as e:
            print(f"  {precision}: Failed — {e}")
else:
    print("TensorRT not available.")
    print("In production: FP16 ~2x speedup | INT8 ~3-4x speedup")

TensorRT 11.0.0.114 available! Converting...
  fp32: models/tensorrt/failure_classifier_fp32.engine (254.4 MB)
  Note: TRT 11.0.0.114 removed FP16 flag — building FP32
  fp16: models/tensorrt/failure_classifier_fp16.engine (254.4 MB)
  Note: TRT 11.0.0.114 removed INT8 flag — building FP32
  int8: models/tensorrt/failure_classifier_int8.engine (254.4 MB)


## 7. Focal Loss Demo

In [16]:
from app.models.failure_classifier.losses import FocalLoss

focal_loss = FocalLoss(num_classes=6, gamma=2.0, label_smoothing=0.1)

# Simulate imbalanced batch: mostly no_failure (class 0), few hallucinations (class 3)
dummy_logits = torch.randn(16, 6).cuda()
dummy_labels = torch.tensor([0,0,0,0,0,0,0,0,0,0,1,1,2,3,4,5]).cuda()  # 10 no_failure, 6 failures

loss = focal_loss(dummy_logits, dummy_labels)
print(f"Focal Loss: {loss.item():.4f}")
print(f"  gamma={focal_loss.gamma} (down-weights easy examples)")
print(f"  label_smoothing={focal_loss.label_smoothing} (prevents overconfidence)")
print(f"  alpha weights per class: {focal_loss.alpha.cpu().numpy().round(3)}")

# Compare with standard CrossEntropy
ce_loss = torch.nn.CrossEntropyLoss()(dummy_logits, dummy_labels)
print(f"\nStandard CrossEntropy: {ce_loss.item():.4f}")
print(f"Focal Loss focuses more on hard/rare examples → better for imbalanced failure classes")

Focal Loss: 1.9022
  gamma=2.0 (down-weights easy examples)
  label_smoothing=0.1 (prevents overconfidence)
  alpha weights per class: [1. 1. 1. 1. 1. 1.]

Standard CrossEntropy: 2.3608
Focal Loss focuses more on hard/rare examples → better for imbalanced failure classes


## 8. Summary

In [17]:
print(f"""{'='*60}
  FaithTrace GPU Inference Pipeline — Summary
{'='*60}

GPU Profile:
  Device:     {profile.name}
  VRAM:       {profile.vram_total_gb} GB total / {profile.vram_available_gb} GB free
  Precision:  {profile.recommended_precision} (recommended)
  Batch Size: {profile.max_batch_size} (max recommended)

Model:
  Architecture:  DistilBERT + RAGAS fusion (773-dim) + 6-class head
  Parameters:    {sum(p.numel() for p in model.parameters()):,} total
  Trainable:     {sum(p.numel() for p in model.parameters() if p.requires_grad):,}
  Loss:          Focal Loss (gamma=2.0, label_smoothing=0.1)

Export:
  ONNX:          {onnx_path} ({os.path.getsize(onnx_path)/1024/1024:.1f} MB)
  Dynamic Batch: batch_size=1 (DistilBERT attention reshapes traced at export time)
  TensorRT:      {'Available' if TRT_AVAILABLE else 'Not available (graceful fallback)'}

Benchmark (batch_size=1, {n_runs} runs):
  PyTorch p50:      {pt_times[n_runs//2]:.2f} ms
  ONNX Runtime p50: {ort_times[n_runs//2]:.2f} ms
  Speedup:          {pt_times[n_runs//2]/ort_times[n_runs//2]:.1f}x

{'='*60}""")

  FaithTrace GPU Inference Pipeline — Summary

GPU Profile:
  Device:     Tesla T4
  VRAM:       15.0 GB total / 14.6 GB free
  Precision:  fp16 (recommended)
  Batch Size: 16 (max recommended)

Model:
  Architecture:  DistilBERT + RAGAS fusion (773-dim) + 6-class head
  Parameters:    66,563,078 total
  Trainable:     38,211,590
  Loss:          Focal Loss (gamma=2.0, label_smoothing=0.1)

Export:
  ONNX:          models/failure_classifier.onnx (254.0 MB)
  Dynamic Batch: batch_size=1 (DistilBERT attention reshapes traced at export time)
  TensorRT:      Available

Benchmark (batch_size=1, 100 runs):
  PyTorch p50:      20.26 ms
  ONNX Runtime p50: 20.95 ms
  Speedup:          1.0x

